In [0]:
from pyspark.sql.functions import *

In [0]:
df_silver = spark.table("uk_ecommerce.silver.online_retail")

In [0]:
display(df_silver)

## Check for cancellations

- Verify whether cancelled invoices, after removing the C prefix, have a corresponding original invoice number in the non-cancelled orders.

In [0]:
# Create cancellation orders DF
cancellations_orders = df_silver.filter(df_silver["InvoiceNo"].startswith("C"))
cancellations_orders = cancellations_orders.withColumn("InvoiceNoWithoutC", regexp_replace("InvoiceNo", "^C", "")).distinct()

# Create DF without cancellations
orders_without_cancellations = df_silver.filter(~df_silver["InvoiceNo"].startswith("C")).distinct()

# Check whether the invoice numbers from cancellations_orders, after removing the C prefix, exist in orders_without_cancellations
joined_tables = cancellations_orders.join(orders_without_cancellations, cancellations_orders["InvoiceNoWithoutC"] == orders_without_cancellations["InvoiceNo"], "left").select(cancellations_orders["InvoiceNoWithoutC"], orders_without_cancellations["InvoiceNo"]).distinct()

display(joined_tables.limit(10))

A total of 3,654 cancellation invoices were identified. For none of them was a non-cancelled invoice found with the same number after removing the C prefix. Therefore, the dataset does not allow us to conclude that an original sale is retained as a second invoice with the same number, nor does it allow us to link a cancellation to its original sale based solely on the InvoiceNo.

- Find a case where one of the items in an order was cancelled, and print all the lines from that invoice, including both the cancelled and non-cancelled items.

In [0]:
cancellations_orders = df_silver.filter(df_silver["InvoiceNo"].startswith("C"))
orders_without_cancellations = df_silver.filter(~df_silver["InvoiceNo"].startswith("C"))

In [0]:
c = (
    cancellations_orders
    .withColumn("cancellation_line_id", monotonically_increasing_id())
    .alias("c")
)

s = orders_without_cancellations.alias("s")

candidate_matches = (
    c.join(
        s,
        (
            (c["CustomerID"] == s["CustomerID"]) &
            (c["StockCode"] == s["StockCode"]) &
            (c["InvoiceDate"] > s["InvoiceDate"]) &
            (abs(c["Quantity"]) <= s["Quantity"])
        ),
        "inner"
    )
    .select(
        c["cancellation_line_id"],

        c["InvoiceNo"].alias("cancellation_invoice_no"),
        c["CustomerID"].alias("customer_id"),
        c["StockCode"].alias("stock_code"),
        c["Description"].alias("cancelled_description"),
        c["Quantity"].alias("cancelled_quantity"),
        c["UnitPrice"].alias("cancelled_unit_price"),
        c["InvoiceDate"].alias("cancellation_date"),

        s["InvoiceNo"].alias("original_invoice_no"),
        s["Description"].alias("original_description"),
        s["Quantity"].alias("original_quantity"),
        s["UnitPrice"].alias("original_unit_price"),
        s["InvoiceDate"].alias("original_sale_date")
    )
    .withColumn(
        "days_between_sale_and_cancellation",
        datediff(
            col("cancellation_date"),
            col("original_sale_date")
        )
    )
)

In [0]:
from pyspark.sql.window import Window

window_closest_sale = Window.partitionBy(
    "cancellation_line_id"
).orderBy(
    col("days_between_sale_and_cancellation").asc(),
    col("original_sale_date").desc()
)

closest_sale_candidates = (
    candidate_matches
    .withColumn(
        "candidate_rank",
        row_number().over(window_closest_sale)
    )
    .filter(col("candidate_rank") == 1)
)

display(
    closest_sale_candidates
    .orderBy("days_between_sale_and_cancellation")
)

In [0]:
original_invoice = orders_without_cancellations.filter(
    col("InvoiceNo") == "542724"
)

cancellation_invoice = cancellations_orders.filter(
    col("InvoiceNo") == "C542913"
)

display(original_invoice.orderBy("StockCode"))
display(cancellation_invoice.orderBy("StockCode"))

A possible partial cancellation was identified. The original invoice contained 13 different products. Only two products appeared in the cancellation invoice: StockCode 21041, with 6 units purchased and 1 unit cancelled, and StockCode 22197, with 36 units purchased and 2 units cancelled. The remaining products from the original invoice did not appear in the cancellation record.

## Metrics Calculation

In [0]:
sales_by_customer = df_silver.groupBy("CustomerID").agg(
    round(sum("totalSales"), 2).alias("TotalSales"),
    count("InvoiceNo").alias("TotalPurchases"),
    round(avg("TotalSales"), 2).alias("AveragePurchase"))

In [0]:
sales_per_product = df_silver.groupBy("StockCode").agg(
    round(sum("TotalSales"), 2).alias("TotalSales"),
    round(sum("Quantity"), 2).alias("TotalQuantity")
    )

In [0]:
sales_per_country = df_silver.groupBy("Country").agg(
    round(sum("TotalSales"), 2).alias("TotalSales"),
    round(sum("Quantity"), 2).alias("TotalQuantity")
)

In [0]:
sales_per_country.write.mode("overwrite").saveAsTable("uk_ecommerce.gold.sales_per_country")
sales_per_product.write.mode("overwrite").saveAsTable("uk_ecommerce.gold.sales_per_product")
sales_by_customer.write.mode("overwrite").saveAsTable("uk_ecommerce.gold.sales_by_customer")